# **Lab1: Regression**
In *lab 1*, you need to finish:

1.  Basic Part: Implement the regression model to predict people's grip force from their weight.
You can use either Matrix Inversion or Gradient Descent.


> *   Step 1: Split Data
> *   Step 2: Preprocess Data
> *   Step 3: Implement Regression
> *   Step 4: Make Prediction
> *   Step 5: Train Model and Generate Result

2.  Advanced Part: Implementing a regression model to predict grip force in a different way (for example, with more variables) than the basic part




---
# 1. Basic Part (50%)
In the first part, you need to implement the regression to predict grip force

Please save the prediction result in a CSV file and submit it to Kaggle

### Import Packages

> Note: You **cannot** import any other package


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import csv
import math
import random

### Global attributes
Define the global attributes\
You can also add your own global attributes here

In [2]:
training_dataroot = 'lab1_basic_training.csv' # Training data file file named as 'lab1_basic_training.csv'
testing_dataroot = 'lab1_basic_testing.csv'   # Testing data file named as 'lab1_basic_testing.csv'
output_dataroot = 'lab1_basic.csv' # Output file will be named as 'lab1_basic.csv'

training_datalist =  [] # Training datalist, saved as numpy array
testing_datalist =  [] # Testing datalist, saved as numpy array

output_datalist =  [] # Your prediction, should be a list with 100 elements

### Load the Input File
First, load the basic input file **lab1_basic_training.csv** and **lab1_basic_testing.csv**

Input data would be stored in *training_datalist* and *testing_datalist*

In [3]:
# Read input csv to datalist
with open(training_dataroot, newline='') as csvfile:
  training_datalist = pd.read_csv(training_dataroot).to_numpy()

with open(testing_dataroot, newline='') as csvfile:
  testing_datalist = pd.read_csv(testing_dataroot).to_numpy()

### Implement the Regression Model

> Note: It is recommended to use the functions we defined, you can also define your own functions

#### Step 1: Split Data
Split data in *training_datalist* into training dataset and validation dataset


In [4]:
def SplitData(data, split_ratio):
    """
    Splits the given dataset into training and validation sets based on the specified split ratio.

    Parameters:
    - data (numpy.ndarray): The dataset to be split. It is expected to be a 2D array where each row represents a data point and each column represents a feature.
    - split_ratio (float): The ratio of the data to be used for training. For example, a value of 0.8 means 80% of the data will be used for training and the remaining 20% for validation.

    Returns:
    - training_data (numpy.ndarray): The portion of the dataset used for training.
    - validation_data (numpy.ndarray): The portion of the dataset used for validation.

    """
    training_data = []
    validation_data = []
    
    # TODO
    # Shuffle the data
    np.random.shuffle(data) # make the data index random

    # Determine the split point
    split_index = int(len(data) * split_ratio)
    
     # Split the data into training and validation sets
    training_data = data[:split_index]
    validation_data = data[split_index:]

    return training_data, validation_data



#### Step 2: Preprocess Data
Handle unreasonable data and missing data

> Hint 1: Outliers and missing data can be addressed by either removing them or replacing them using statistical methods (e.g., the mean of all data).

> Hint 2: Missing data are represented as `np.nan`, so functions like `np.isnan()` can be used to detect them.

> Hint 3: Methods such as the Interquartile Range (IQR) can help detect outliers

In [5]:
def PreprocessData(data):
    """
    Preprocess the given dataset and return the result.

    Parameters:
    - data (numpy.ndarray): The dataset to preprocess. It is expected to be a 2D array where each row represents a data point and each column represents a feature.

    Returns:
    - preprocessedData (numpy.ndarray): Preprocessed data.
    """
    preprocessedData = []

    # TODO
    # Remove rows with missing data (NaN)
    data = data[~np.isnan(data).any(axis=1)]

    # Remove negative values and zeros
    data = data[(data > 0).all(axis=1)]

    # Replace outliers using IQR
    # Initialize preprocessedData with a copy of the data
    preprocessedData = np.copy(data)
    
    for col in range(data.shape[1]):
        # Get the 25th and 75th percentiles (Q1 and Q3)
        Q1 = np.percentile(data[:, col], 25)
        Q3 = np.percentile(data[:, col], 75)
        IQR = Q3 - Q1
        
        # Define the lower and upper bounds for outliers
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Detect outliers
        outliers_mask = (data[:, col] < lower_bound) | (data[:, col] > upper_bound)
        
        # Replace outliers with the column's median (you can choose mean or other values)
        col_median = np.median(data[:, col][~outliers_mask])
        preprocessedData[outliers_mask, col] = col_median


    return preprocessedData



### Step 3: Implement Regression
You have to use Gradient Descent to finish this part

In [6]:
def Regression(dataset):
    """
    Performs regression on the given dataset and return the coefficients.

    Parameters:
    - dataset (numpy.ndarray): A 2D array where each row represents a data point.

    Returns:
    - w (numpy.ndarray): The coefficients of the regression model. For example, y = w[0] + w[1] * x + w[2] * x^2 + ...
    """

    X = dataset[:, :1] # X is the first column of the dataset
    y = dataset[:, 1] # y is the last column of the dataset

    # Decide on the degree of the polynomial
    degree = 4 # For example, quadratic regression

    # Simple scaling: Divide X by 20
    scale_factor = 20
    X_scaled = X / scale_factor

    # Add polynomial features to X
    X_poly = np.ones((X_scaled.shape[0], 1))  # Add intercept term (column of ones)
    for d in range(1, degree + 1):
        X_poly = np.hstack((X_poly, X_scaled ** d))  # Add x^d terms to feature matrix
    
    # Initialize coefficients (weights) to small random values
    num_dimensions = X_poly.shape[1]  # Number of features (including intercept and polynomial terms)
    w = np.random.randn(num_dimensions) * 0.01

    # Set hyperparameters
    num_iteration = 100000
    learning_rate = 0.00001

    # Gradient Descent
    m = len(y)  # Number of data points
    for iteration in range(num_iteration):
        # Prediction using current weights and compute error
        y_pred = np.dot(X_poly, w) #calculate the predicted y value
        error = y_pred - y

        # Compute gradient
        gradient = (1/m) * np.dot(X_poly.T, error)

        # Update the weights
        w -= learning_rate * gradient

        # Optionally, print the cost every 100 iterations
        if iteration % 10000 == 0:
            cost = (1/(2*m)) * np.sum(error**2)
            print(f"Iteration {iteration}, Cost: {cost}")

    # Adjust the weights to account for the scaling
    w_adjusted = np.zeros_like(w)
    w_adjusted[0] = w[0]
    for i in range(1, len(w)):
        w_adjusted[i] = w[i] / (scale_factor ** i)

    return w_adjusted



### Step 4: Make Prediction
Make prediction of testing dataset and store the value in *output_datalist*

In [7]:

def MakePrediction(w, test_dataset):
    """
    Predicts the output for a given test dataset using a regression model.

    Parameters:
    - w (numpy.ndarray): The coefficients of the model, where each element corresponds to
                               a coefficient for the respective power of the independent variable.
    - test_dataset (numpy.ndarray): A 1D array containing the input values (independent variable)
                                          for which predictions are to be made.

    Returns:
    - list/numpy.ndarray: A list or 1d array of predicted values corresponding to each input value in the test dataset.
    """
    prediction = []

    # TODO
    # Add polynomial features to the test dataset
    degree = len(w) - 1  # The degree of the polynomial model is determined by the length of w
    X_poly = np.ones((test_dataset.shape[0], 1))  # Start with the intercept term (column of ones)

    # Add columns corresponding to x, x^2, ..., x^degree
    for d in range(1, degree + 1):
        X_poly = np.hstack((X_poly, test_dataset.reshape(-1, 1) ** d))

    # Step 2: Make predictions using matrix multiplication (dot product)
    prediction = X_poly.dot(w)
    
    return prediction



### Step 5: Train Model and Generate Result

Use the above functions to train your model on training dataset, and predict the answer of testing dataset.

Save your predicted values in `output_datalist`

> Notice: **Remember to inclue the coefficients of your model in the report**



In [8]:
# (1) Split data
training_data, validation_data = SplitData(training_datalist, 0.90)

# (2) Preprocess data
training_data_cleaned = PreprocessData(training_data)
validation_data_cleaned = PreprocessData(validation_data)

# (3) Train regression model
w = Regression(training_data_cleaned)
print(w)

# (4) Predict validation dataset's answer, calculate MAPE comparing to the ground truth
def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100
validation_X = validation_data_cleaned[:, 0:1]  # Input (independent variable)
validation_y = validation_data_cleaned[:, 1]    # Ground truth (dependent variable)
validation_predict = MakePrediction(w, validation_X)
validation_mape = MAPE(validation_y, validation_predict)
print(f"Validation MAPE: {validation_mape:.2f}%")

# (5) Make prediction of testing dataset and store the values in output_datalist
test_X = testing_datalist[:, 0:1]  # Extract test input (X)
output_datalist = MakePrediction(w, test_X) # Make predictions using the trained model


Iteration 0, Cost: 1056.4794936244364
Iteration 10000, Cost: 41.41284729201931
Iteration 20000, Cost: 41.37850280461506
Iteration 30000, Cost: 41.34942564587343
Iteration 40000, Cost: 41.321535679753985
Iteration 50000, Cost: 41.294784342267384
Iteration 60000, Cost: 41.26912513963632
Iteration 70000, Cost: 41.24451347657343
Iteration 80000, Cost: 41.22090657875794
Iteration 90000, Cost: 41.19826341847989
[ 3.94763677e-01  4.04485181e-02  3.73054083e-03  2.45802696e-04
 -2.39411858e-06]
Validation MAPE: 17.75%


### *Write the Output File*

Write the prediction to output csv and upload the file to Kaggle
> Format: 'Id', 'gripForce'


In [9]:
# Assume that output_datalist is a list (or 1d array) with length = 100

with open(output_dataroot, 'w', newline='', encoding="utf-8") as csvfile:
  writer = csv.writer(csvfile)
  writer.writerow(['Id', 'gripForce'])
  for i in range(len(output_datalist)):
    writer.writerow([i,output_datalist[i]])


# 2. Advanced Part (45%)
In the second part, you need to implement regression differently from the basic part to improve your grip force predictions. You must use more than two features.

You can choose either matrix inversion or gradient descent for this part

We have provided `lab1_advanced_training.csv` for your training

> Notice: Be cautious of the "gender" attribute, as it is represented by "F"/"M" rather than a numerical value.

Please save the prediction result in a CSV file and submit it to Kaggle

In [10]:
training_dataroot = 'lab1_advanced_training.csv' # Training data file file named as 'lab1_advanced_training.csv'
testing_dataroot = 'lab1_advanced_testing.csv'   # Testing data file named as 'lab1_advanced_testing.csv'
output_dataroot = 'lab1_advanced.csv' # Output file will be named as 'lab1_advanced.csv'

training_datalist =  [] # Training datalist, saved as numpy array
testing_datalist =  [] # Testing datalist, saved as numpy array

output_datalist =  [] # Your prediction, should be a list with 3000 elements

In [11]:
# Read input csv to datalist
with open(training_dataroot, newline='') as csvfile:
  training_datalist = pd.read_csv(training_dataroot).to_numpy()

with open(testing_dataroot, newline='') as csvfile:
  testing_datalist = pd.read_csv(testing_dataroot).to_numpy()

In [12]:
def gender_to_number(gender):
    if gender == 'F': return 0
    elif gender == 'M': return 1
    else: return -1


def SplitData2(data, split_ratio):
    training_data = []
    validation_data = []
    
    # np.random.shuffle(data)
    split_index = int(len(data) * split_ratio)
    training_data = data[:split_index]
    validation_data = data[split_index:]

    return training_data, validation_data


def PreprocessData2(data):
    data[:, 1] = np.vectorize(gender_to_number)(data[:, 1])
    data = data.astype(float) #to prevent type error

    # Remove rows with missing data (NaN)
    data = data[~np.isnan(data).any(axis=1)]

    # Remove negative values and zeros
    data = data[(data >= 0).all(axis=1)]

    # Replace outliers using IQR
    # Initialize preprocessedData with a copy of the data
    preprocessedData = np.copy(data)
    
    for col in range(data.shape[1]):
        # Get the 25th and 75th percentiles (Q1 and Q3)
        Q1 = np.percentile(data[:, col], 25)
        Q3 = np.percentile(data[:, col], 75)
        IQR = Q3 - Q1
        
        # Define the lower and upper bounds for outliers
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Detect outliers
        outliers_mask = (data[:, col] < lower_bound) | (data[:, col] > upper_bound)
        
        # Replace outliers with the column's median (you can choose mean or other values)
        col_median = np.median(data[:, col][~outliers_mask])
        preprocessedData[outliers_mask, col] = col_median


    return preprocessedData


def Regression2(dataset):
    X = dataset[:, :7] # X is the first seven column of the dataset
    y = dataset[:, 7] # y is the last column of the dataset

    # Decide on the degree of the polynomial
    degree = 5 

    # Normalize X to prevent numerical instability
    X_mean = np.mean(X)
    X_std = np.std(X)
    X_normalized = (X - X_mean) / X_std

    # Add polynomial features to X
    X_poly = np.ones((X_normalized.shape[0], 1))  # Add intercept term (column of ones)
    for d in range(1, degree + 1):
        X_poly = np.hstack((X_poly, X_normalized ** d))  # Add x^d terms to feature matrix

    # Initialize coefficients (weights) to small random values
    num_dimensions = X_poly.shape[1]  # Number of features (including intercept and polynomial terms)
    w = np.random.randn(num_dimensions) * 0.01

    # Set hyperparameters
    num_iteration = 2000000
    learning_rate = 0.004

    # Gradient Descent
    m = len(y)  # Number of data points
    for iteration in range(num_iteration):
        # Prediction using current weights and compute error
        y_pred = np.dot(X_poly, w)  # Calculate the predicted y value
        error = y_pred - y

        # Compute gradient
        gradient = (1/m) * np.dot(X_poly.T, error)

        # Update the weights
        w -= learning_rate * gradient

        # Optionally, print the cost every 10000 iterations
        if iteration % 10000 == 0:
            cost = (1/(2*m)) * np.sum(error**2)
            print(f"Iteration {iteration}, Cost: {cost}")

    return w


def MakePrediction2(w, test_dataset):
    # Ensure test_dataset is 2D, even if it's a single feature (add an axis if necessary)
    if test_dataset.ndim == 1:
        test_dataset = test_dataset[:, np.newaxis]  # Convert 1D array to 2D
    
    # Degree of the polynomial model
    degree = 5

    # Normalize X to prevent numerical instability
    X_mean = np.mean(test_dataset)
    X_std = np.std(test_dataset)
    X_normalized = (test_dataset - X_mean) / X_std

    # Add polynomial features to the test dataset
    X_poly = np.ones((X_normalized.shape[0], 1))  # Start with the intercept term (column of ones)

    # Add polynomial terms for each feature
    for d in range(1, degree + 1):
        X_poly = np.hstack((X_poly, X_normalized ** d))  # Add x^d terms for each feature in the dataset

    # Make predictions using matrix multiplication (dot product)
    prediction = X_poly.dot(w)
    
    return prediction

In [13]:
# (1) Split data
training_data, validation_data = SplitData(training_datalist, 0.90)

# (2) Preprocess data
training_data_cleaned = PreprocessData2(training_data)
validation_data_cleaned = PreprocessData2(validation_data)
testing_datalist[:, 1] = np.vectorize(gender_to_number)(testing_datalist[:, 1])
testing_data_cleaned = testing_datalist.astype(float) #to prevent type error

# (3) Train regression model
w = Regression2(training_data_cleaned)
print(w)

# (4) Predict validation dataset's answer, calculate MAPE comparing to the ground truth
def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100
validation_X = validation_data_cleaned[:, :7]  # Input (independent variable)
validation_y = validation_data_cleaned[:, -1]    # Ground truth (dependent variable)
validation_predict = MakePrediction2(w, validation_X)
validation_mape = MAPE(validation_y, validation_predict)
print(f"Validation MAPE: {validation_mape:.2f}%")

# (5) Make prediction of testing dataset and store the values in output_datalist
test_X = testing_data_cleaned[:, :7]  # Extract test input (X)
output_datalist = MakePrediction2(w, test_X) # Make predictions using the trained model
with open(output_dataroot, 'w', newline='', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Id', 'gripForce'])
    for i in range(len(output_datalist)):
        writer.writerow([i,output_datalist[i]])

Iteration 0, Cost: 1055.2781930431302
Iteration 10000, Cost: 27.49781368612223
Iteration 20000, Cost: 26.215763446692307
Iteration 30000, Cost: 25.66239054738172
Iteration 40000, Cost: 25.331869255535373
Iteration 50000, Cost: 25.09764953988275
Iteration 60000, Cost: 24.91329515917599
Iteration 70000, Cost: 24.759250401150755
Iteration 80000, Cost: 24.626115795255803
Iteration 90000, Cost: 24.508680741252025
Iteration 100000, Cost: 24.403667637207217
Iteration 110000, Cost: 24.3088149764605
Iteration 120000, Cost: 24.22246519087927
Iteration 130000, Cost: 24.143355590986747
Iteration 140000, Cost: 24.07049891624498
Iteration 150000, Cost: 24.00310813530882
Iteration 160000, Cost: 23.940545750120684
Iteration 170000, Cost: 23.882288054989118
Iteration 180000, Cost: 23.82789919927019
Iteration 190000, Cost: 23.777011994760926
Iteration 200000, Cost: 23.729313513379562
Iteration 210000, Cost: 23.68453416062704
Iteration 220000, Cost: 23.64243930937692
Iteration 230000, Cost: 23.6028228410

# Save the Code File
Please save your code and submit it as an ipynb file! (**Lab1.ipynb**)